In [1]:
from pynq import Overlay, allocate
import numpy as np, time, math

In [2]:
BIT_PATH = "/home/xilinx/matmul_opt.bit"   # matmul_opt.hwh must be in the same folder

In [3]:
M, N, K = 64, 64, 64

In [4]:
ol = Overlay(BIT_PATH)  # HWH auto-loaded from same path/name
ol.download()

In [5]:
print("Loaded overlay. IPs found:", list(ol.ip_dict.keys()))

Loaded overlay. IPs found: ['matmul_0', 'processing_system7_0']


In [6]:
ip_name = None
for k in ol.ip_dict.keys():
    if "matmul" in k:
        ip_name = k
        break
if ip_name is None:
    for k in ol.ip_dict.keys():
        if "processing_system7" not in k:
            ip_name = k
            break

ip = getattr(ol, ip_name)
print("Using IP:", ip_name)
print(ip.register_map)

Using IP: matmul_0
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  Matrix_A_DRAM_1 = Register(Matrix_A_DRAM=write-only),
  Matrix_A_DRAM_2 = Register(Matrix_A_DRAM=write-only),
  Matrix_B_DRAM_1 = Register(Matrix_B_DRAM=write-only),
  Matrix_B_DRAM_2 = Register(Matrix_B_DRAM=write-only),
  Matrix_C_DRAM_1 = Register(Matrix_C_DRAM=write-only),
  Matrix_C_DRAM_2 = Register(Matrix_C_DRAM=write-only)
}


In [7]:
def write_ptr64(ip, base_name, addr):
    low  = int(addr & 0xFFFFFFFF)
    high = int((addr >> 32) & 0xFFFFFFFF)
    setattr(ip.register_map, base_name + "_1", low)
    setattr(ip.register_map, base_name + "_2", high)

rm = ip.register_map
ptr_fields = [f for f in dir(rm) if f.startswith("Matrix_") and (f.endswith("_1") or f.endswith("_2"))]
print("Pointer fields present:", ptr_fields)

Pointer fields present: ['Matrix_A_DRAM_1', 'Matrix_A_DRAM_2', 'Matrix_B_DRAM_1', 'Matrix_B_DRAM_2', 'Matrix_C_DRAM_1', 'Matrix_C_DRAM_2']


In [8]:
A = allocate((M, N), dtype=np.int16)
B = allocate((N, K), dtype=np.int16)
C = allocate((M, K), dtype=np.int16)

rng = np.random.default_rng(42)   # same seed as the unoptimized notebook
A[:] = rng.integers(low=-100, high=101, size=A.shape, dtype=np.int16)
B[:] = rng.integers(low=-100, high=101, size=B.shape, dtype=np.int16)
C.fill(0)

print("A.shape, B.shape, C.shape:", A.shape, B.shape, C.shape)

A.shape, B.shape, C.shape: (64, 64) (64, 64) (64, 64)


In [9]:
def matmul_cpu_trunc16(A_in, B_in):
    # slow pure-Python loop, 16-bit truncation like the HLS code
    Cref = np.zeros((M, K), dtype=np.int16)
    for i in range(M):
        for j in range(K):
            acc = np.int16(0)
            for d in range(N):
                prod = np.int32(A_in[i, d]) * np.int32(B_in[d, j])
                acc = np.int16(np.int32(acc) + np.int16(prod))
            Cref[i, j] = acc
    return Cref

# int32 accumulate, cast to int16 at the end. Same result as trunc16 (arithmetic is mod 2^16), but fast.
def numpy_int32_acc(A_in, B_in):
    return (A_in.astype(np.int32) @ B_in.astype(np.int32)).astype(np.int16)

def time_fn_ms(fn, runs=5, warmup=1):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(times)), times

In [10]:
RUN_SLOW_LOOP = False   # True = also time the pure-Python loop (about 22 s on the board)
if RUN_SLOW_LOOP:
    t0 = time.perf_counter()
    C_cpu_exact = matmul_cpu_trunc16(A, B)
    cpu_ms_exact = (time.perf_counter() - t0) * 1e3
    print(f"CPU (trunc16 loop, 1 run): {cpu_ms_exact:.2f} ms")
else:
    C_cpu_exact = numpy_int32_acc(A, B)

cpu_ms_numpy, cpu_runs_numpy = time_fn_ms(lambda: numpy_int32_acc(A, B), runs=5, warmup=1)
print(f"PS computation completed in {cpu_ms_numpy:.2f} ms (NumPy, median of 5)")

PS computation completed in 4.05 ms (NumPy, median of 5)


In [11]:
write_ptr64(ip, "Matrix_A_DRAM", A.physical_address)
write_ptr64(ip, "Matrix_B_DRAM", B.physical_address)
write_ptr64(ip, "Matrix_C_DRAM", C.physical_address)

In [12]:
def run_kernel(timeout_s=5):
    t0 = time.perf_counter()
    ip.write(0x00, 1)                      # ap_start
    while (ip.read(0x00) & 0x2) == 0:      # wait AP_DONE
        if time.perf_counter() - t0 > timeout_s:
            print("TIMEOUT, CTRL =", hex(ip.read(0x00)))
            break
    return (time.perf_counter() - t0) * 1e3

run_kernel()   # warm-up

fpga_times = []
for _ in range(20):
    C.fill(0)
    fpga_times.append(run_kernel())

fpga_ms = float(np.median(fpga_times))
print("FPGA times (ms):", [f"{x:.3f}" for x in fpga_times[:5]], "...")
print(f"Kernel completed in {fpga_ms:.3f} ms (median of 20)")

FPGA times (ms): ['0.432', '0.389', '0.408', '0.404', '0.393'] ...
Kernel completed in 0.402 ms (median of 20)


In [13]:
C_fpga = C.reshape(M, K)
max_abs = int(np.max(np.abs(C_fpga.astype(np.int32) - C_cpu_exact.astype(np.int32))))
print("Max |HW-CPU| =", max_abs)
print("PASS: FPGA output matches CPU output!" if max_abs == 0 else "FAIL: mismatch")

ops = 2.0 * M * N * K
cpu_ms_for_speedup = cpu_ms_numpy
cpu_gflops  = (ops / (cpu_ms_for_speedup/1e3)) / 1e9
fpga_gflops = (ops / (fpga_ms/1e3)) / 1e9
speedup = cpu_ms_for_speedup / fpga_ms

print("\n=== Performance (optimized) ===")
print(f"CPU time (used for speedup): {cpu_ms_for_speedup:.3f} ms")
print(f"FPGA time (kernel only)    : {fpga_ms:.3f} ms")
print(f"CPU perf : {cpu_gflops:.3f} GFLOP/s")
print(f"FPGA perf: {fpga_gflops:.3f} GFLOP/s")
print(f"Speedup (CPU/FPGA) = {speedup:.2f}x")

Max |HW-CPU| = 0
PASS: FPGA output matches CPU output!

=== Performance (optimized) ===
CPU time (used for speedup): 4.047 ms
FPGA time (kernel only)    : 0.402 ms
CPU perf : 0.130 GFLOP/s
FPGA perf: 1.305 GFLOP/s
Speedup (CPU/FPGA) = 10.08x


In [14]:
def peek_matrix(name, X, corner=8):
    m, n = X.shape
    c = min(corner, m, n)
    print(f"\n{name} shape={X.shape} (top-left {c}x{c})")
    print(X[:c, :c])

peek_matrix("A", A)
peek_matrix("B", B)
peek_matrix("C_fpga", C_fpga)
peek_matrix("C_cpu", C_cpu_exact)


A shape=(64, 64) (top-left 8x8)
[[-70 -83  97  55 100  31 -33 -12]
 [-56 -82 -47  11  50  78 -35 -88]
 [ -2 -18 -42 -35 -62  82 -11 -26]
 [ 32 -52 -95  37  65  27  64 -72]
 [-95  15  80  27 -39  13 -24  11]
 [ 73 100 -10  33 -12 -18  94 -19]
 [-55  22  50 -28 -17  92 -55 -83]
 [ 71  45 -66 -39 -22  62  47  16]]

B shape=(64, 64) (top-left 8x8)
[[  -4   32 -100  -66  -31  -40   32  -28]
 [ -82  -48  -35   24   34    9  -74  -98]
 [  82  -52   49    2  -60   15   93   39]
 [ -11   18  -65   20   88  -17   70   44]
 [  -4  -10   25  -90   -9  -42   40   -6]
 [ -73  -67   40   76   55    6   21   58]
 [  33  -91  -10   73  -78  -96   42   28]
 [  -8    8  -65  -64  -15  -33  -58   61]]

C_fpga shape=(64, 64) (top-left 8x8)
[[  3807   6361  10260   4221   6453  25834  30080   4882]
 [-16197  12293  32511  10414    395 -29689  15833   6837]
 [   -56 -31723 -23367  17999   3512   -849   4917  10130]
 [ 13378 -26437  25731   5710   7768  -7832  15899   4255]
 [ 11069 -17915   7753  14209   30

In [15]:
for buf in (A, B, C):
    buf.freebuffer()
ol.free()